# WRDS Stock Lab Demo

## Analytical Problem

This project aims to analyze stock performance using WRDS data, focusing on return patterns, volatility, and risk.

## Target User

The intended users are finance students and beginner investors.

## Key Questions

- How do stock returns behave over time?
- What is the relationship between risk and return?
- How does cumulative performance differ across stocks?

This notebook shows the Python-side workflow behind the Streamlit app:

1. connect to WRDS
2. run stock SQL
3. inspect summary outputs
4. compare multiple stocks with cumulative returns and drawdowns


## Imports

In [ ]:
from getpass import getpass

import pandas as pd

from analysis import (
    build_cumulative_return_series,
    build_drawdown_series,
    build_drawdown_summary,
    build_normalized_price_series,
    build_ticker_summary,
    detect_stock_columns,
    summarize_stock_data,
)
from data_loader import get_wrds_connection, run_wrds_sql


## Connect To WRDS

If your local WRDS authentication is already configured through `.pgpass`, you can omit the password.

In [ ]:
username = input("WRDS username: ").strip()
password = getpass("WRDS password (leave blank if using .pgpass): ")

db = get_wrds_connection(
    username=username or None,
    password=password or None,
)


## Build A Sample Stock Query

In [ ]:
watchlist = ["AAPL", "MSFT", "NVDA", "AMZN", "META"]
watchlist_sql = ", ".join(f"'{ticker}'" for ticker in watchlist)

sql = f"""
select
    a.date,
    b.ticker,
    a.permno,
    abs(a.prc) as prc,
    a.ret,
    a.vol,
    a.shrout
from crsp.dsf as a
join crsp.stocknames as b
  on a.permno = b.permno
 and a.date between b.namedt and b.nameenddt
where b.ticker in ({watchlist_sql})
  and a.date between '2024-01-01' and '2024-12-31'
order by a.date, b.ticker
limit 5000
"""

print(sql)


## Run SQL

In [ ]:
df = run_wrds_sql(db, sql)
df.head()


## Inspect Detected Stock Columns

In [ ]:
detect_stock_columns(df)


## Summary

In [ ]:
summary = summarize_stock_data(df)
summary


## Ticker Comparison Table

In [ ]:
ticker_summary = build_ticker_summary(df)
ticker_summary


## Normalized Price Comparison

In [ ]:
normalized_price = build_normalized_price_series(df, focus_tickers=watchlist)
normalized_price.plot(figsize=(12, 5), title="Normalized Price Comparison (Base = 100)")


## Cumulative Return Comparison

In [ ]:
cumulative_return = build_cumulative_return_series(df, focus_tickers=watchlist)
cumulative_return.plot(figsize=(12, 5), title="Cumulative Return Comparison (%)")


## Drawdown Comparison

In [ ]:
drawdown = build_drawdown_series(df, focus_tickers=watchlist)
drawdown.plot(figsize=(12, 5), title="Drawdown Comparison (%)")


## Drawdown Summary

In [ ]:
build_drawdown_summary(df, focus_tickers=watchlist)


## Clean Up

In [ ]:
db.close()
